## Section 1: Setup and Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

matplotlib.use('Agg')
os.makedirs('outputs/eda_plots', exist_ok=True)

df = pd.read_parquet('outputs/final_dataset.parquet')
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names (first 20):")
print(df.columns[:20].tolist())
print(f"\nColumn names (last 5):")
print(df.columns[-5:].tolist())
print(f"\nData types:")
print(df.dtypes.value_counts())

Dataset shape: (23896, 719)

Column names (first 20):
['subject_id', 'hadm_id', 'gender', 'age_at_admission', 'admission_type', 'insurance', 'race', 'discharge_location', 'marital_status', 'los_days', 'came_via_ed', 'prior_admissions_count', 'days_to_next', 'readmitted_30d', "'1    '", "'10   '", "'100  '", "'101  '", "'102  '", "'103  '"]

Column names (last 5):
["'SYM015'", "'SYM016'", "'SYM017'", "'SYM018'", "'XXX000'"]

Data types:
int32     707
object      6
int64       5
Int64       1
Name: count, dtype: int64


## Section 2: Missing Value Analysis

In [2]:
print(f"Total rows: {len(df):,}")
print()

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("No missing values found.")
else:
    missing_df = pd.DataFrame({
        'column': missing.index,
        'missing_count': missing.values,
        'missing_pct': (missing.values / len(df) * 100).round(2)
    })
    print("Columns with missing values:")
    print(missing_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.4)))
    ax.barh(missing_df['column'], missing_df['missing_pct'], color='steelblue')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('outputs/eda_plots/missing_values.png', dpi=150)
    plt.close()
    print("\nSaved: outputs/eda_plots/missing_values.png")

Total rows: 23,896

Columns with missing values:
            column  missing_count  missing_pct
      days_to_next           9783        40.94
discharge_location           6577        27.52
    marital_status            510         2.13
         insurance            403         1.69

Saved: outputs/eda_plots/missing_values.png


## Section 3: Target Variable Distribution

In [3]:
vc = df['readmitted_30d'].value_counts().sort_index()
rate = df['readmitted_30d'].mean() * 100

print("readmitted_30d value counts:")
print(vc)
print(f"\nReadmission rate: {rate:.2f}%")

labels = ['Not Readmitted (0)', 'Readmitted (1)']
counts = [vc.get(0, 0), vc.get(1, 0)]
pcts = [c / len(df) * 100 for c in counts]
colors = ['#4c72b0', '#dd8452']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts, color=colors, width=0.5)
for bar, cnt, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{cnt:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Count')
ax.set_title('30-Day Readmission Distribution')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig('outputs/eda_plots/target_distribution.png', dpi=150)
plt.close()
print("Saved: outputs/eda_plots/target_distribution.png")

readmitted_30d value counts:
readmitted_30d
0    19177
1     4719
Name: count, dtype: int64

Readmission rate: 19.75%
Saved: outputs/eda_plots/target_distribution.png


## Section 4: Clinical Feature Distributions

In [4]:
numeric_cols = ['age_at_admission', 'los_days', 'prior_admissions_count']
numeric_cols = [c for c in numeric_cols if c in df.columns]

for col in numeric_cols:
    s = df[col].dropna().astype(float)
    mean, std = s.mean(), s.std()
    outliers = (s > mean + 3 * std).sum()

    print(f"--- {col} ---")
    print(f"  min   : {s.min():.2f}")
    print(f"  max   : {s.max():.2f}")
    print(f"  mean  : {mean:.2f}")
    print(f"  median: {s.median():.2f}")
    print(f"  std   : {std:.2f}")
    print(f"  values > 3 std above mean: {outliers:,}")
    print()

    fig, ax = plt.subplots(figsize=(8, 4))
    for label, grp in df.groupby('readmitted_30d'):
        vals = grp[col].dropna().astype(float)
        vals.plot.hist(ax=ax, alpha=0.5, bins=40, density=True,
                       label=f"readmitted={int(label)}")
        vals.plot.kde(ax=ax, label='')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {col} by Readmission')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_dist.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_dist.png")

--- age_at_admission ---
  min   : 18.00
  max   : 103.00
  mean  : 58.81
  median: 60.00
  std   : 18.91
  values > 3 std above mean: 0

Saved: outputs/eda_plots/age_at_admission_dist.png
--- los_days ---
  min   : 0.00
  max   : 193.00
  mean  : 4.59
  median: 3.00
  std   : 6.69
  values > 3 std above mean: 466

Saved: outputs/eda_plots/los_days_dist.png
--- prior_admissions_count ---
  min   : 0.00
  max   : 92.00
  mean  : 3.06
  median: 1.00
  std   : 6.36
  values > 3 std above mean: 491

Saved: outputs/eda_plots/prior_admissions_count_dist.png


## Section 5: Categorical Feature Analysis

In [5]:
cat_cols = ['gender', 'admission_type', 'insurance', 'race',
            'discharge_location', 'marital_status']
cat_cols = [c for c in cat_cols if c in df.columns]

for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    pct = (vc / len(df) * 100).round(2)
    summary = pd.DataFrame({'count': vc, 'pct': pct})

    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(summary.to_string())
    print()

    plot_vc = vc.sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, len(plot_vc) * 0.35)))
    ax.barh(plot_vc.index.astype(str), plot_vc.values, color='steelblue')
    ax.set_xlabel('Count')
    ax.set_title(f'{col} - Value Counts')
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_counts.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_counts.png\n")

--- gender (2 unique values) ---
        count    pct
gender              
F       12393  51.86
M       11503  48.14

Saved: outputs/eda_plots/gender_counts.png

--- admission_type (9 unique values) ---
                             count    pct
admission_type                           
EW EMER.                      7788  32.59
EU OBSERVATION                5225  21.87
OBSERVATION ADMIT             3564  14.91
URGENT                        2312   9.68
SURGICAL SAME DAY ADMISSION   1960   8.20
DIRECT OBSERVATION            1079   4.52
DIRECT EMER.                  1028   4.30
ELECTIVE                       584   2.44
AMBULATORY OBSERVATION         356   1.49

Saved: outputs/eda_plots/admission_type_counts.png

--- insurance (5 unique values) ---
           count    pct
insurance              
Medicare   10809  45.23
Private     7489  31.34
Medicaid    4563  19.10
Other        615   2.57
None         403   1.69
No charge     17   0.07

Saved: outputs/eda_plots/insurance_counts.png

--- ra

## Section 6: Readmission Rate by Category

In [6]:
for col in cat_cols:
    rate_df = (
        df.groupby(col, dropna=False)['readmitted_30d']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'readmission_rate', 'count': 'n'})
        .sort_values('readmission_rate', ascending=False)
    )
    rate_df['readmission_rate_pct'] = (rate_df['readmission_rate'] * 100).round(2)

    print(f"--- {col} ---")
    print(rate_df[['n', 'readmission_rate_pct']].to_string())
    print()

    plot_df = rate_df.sort_values('readmission_rate', ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, len(plot_df) * 0.35)))
    ax.barh(plot_df.index.astype(str), plot_df['readmission_rate_pct'], color='#dd8452')
    ax.set_xlabel('Readmission Rate (%)')
    ax.set_title(f'Readmission Rate by {col}')
    plt.tight_layout()
    plt.savefig(f'outputs/eda_plots/{col}_readmission_rate.png', dpi=150)
    plt.close()
    print(f"Saved: outputs/eda_plots/{col}_readmission_rate.png\n")

--- gender ---
            n  readmission_rate_pct
gender                             
M       11503                 20.70
F       12393                 18.87

Saved: outputs/eda_plots/gender_readmission_rate.png

--- admission_type ---
                                n  readmission_rate_pct
admission_type                                         
DIRECT EMER.                 1028                 33.46
ELECTIVE                      584                 30.31
EW EMER.                     7788                 20.81
OBSERVATION ADMIT            3564                 20.03
DIRECT OBSERVATION           1079                 19.74
EU OBSERVATION               5225                 18.64
URGENT                       2312                 14.97
SURGICAL SAME DAY ADMISSION  1960                 14.39
AMBULATORY OBSERVATION        356                 13.48

Saved: outputs/eda_plots/admission_type_readmission_rate.png

--- insurance ---
               n  readmission_rate_pct
insurance                  

## Section 7: CCSR Feature Summary

In [7]:
non_ccsr = {
    'subject_id', 'hadm_id', 'gender', 'age_at_admission',
    'admission_type', 'insurance', 'race', 'discharge_location',
    'marital_status', 'los_days', 'came_via_ed',
    'prior_admissions_count', 'days_to_next', 'readmitted_30d'
}
ccsr_cols = [c for c in df.columns if c not in non_ccsr]

print(f"Total CCSR columns: {len(ccsr_cols)}")

empty_ccsr = [c for c in ccsr_cols if df[c].sum() == 0]
print(f"CCSR columns with all zeros (no patient has this condition): {len(empty_ccsr)}")

prevalence = df[ccsr_cols].mean().sort_values(ascending=False)

print("\nTop 20 most prevalent CCSR categories:")
top20 = prevalence.head(20)
for cat, prev in top20.items():
    print(f"  {cat:<20}  {prev*100:.2f}%")

nonzero_prev = prevalence[prevalence > 0]
bottom10 = nonzero_prev.tail(10)
print("\nBottom 10 least prevalent (nonzero) CCSR categories:")
for cat, prev in bottom10.items():
    print(f"  {cat:<20}  {prev*100:.4f}%")

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(top20)), top20.values * 100, color='steelblue')
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index.tolist(), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Top 20 Most Prevalent CCSR Diagnosis Categories')
plt.tight_layout()
plt.savefig('outputs/eda_plots/ccsr_top20_prevalence.png', dpi=150)
plt.close()
print("\nSaved: outputs/eda_plots/ccsr_top20_prevalence.png")

Total CCSR columns: 705
CCSR columns with all zeros (no patient has this condition): 0

Top 20 most prevalent CCSR categories:
  'XXX000'              58.59%
  '259  '               39.73%
  '98   '               37.20%
  '53   '               31.52%
  'END010'              31.11%
  '257  '               29.90%
  '55   '               29.34%
  'CIR007'              28.95%
  'END011'              28.44%
  '59   '               27.92%
  '95   '               27.41%
  '663  '               27.03%
  '138  '               25.93%
  '657  '               25.27%
  '155  '               25.08%
  'FAC025'              24.78%
  'DIG004'              24.74%
  '58   '               24.36%
  '106  '               22.79%
  'MBD002'              21.54%

Bottom 10 least prevalent (nonzero) CCSR categories:
  'EYE011'              0.0084%
  'INJ015'              0.0084%
  'INJ048'              0.0042%
  'FAC008'              0.0042%
  'NEO005'              0.0042%
  'NEO009'              0.0042%
  'INJ0

## Section 8: Class Imbalance and Correlation Check

In [8]:
from scipy.stats import pointbiserialr

target_vc = df['readmitted_30d'].value_counts()
majority = target_vc.max()
minority = target_vc.min()
print(f"Class balance ratio (majority / minority): {majority / minority:.2f}:1")
print(f"  Class 0: {target_vc.get(0, 0):,}")
print(f"  Class 1: {target_vc.get(1, 0):,}")
print()

num_cols = ['age_at_admission', 'los_days', 'prior_admissions_count']
num_cols = [c for c in num_cols if c in df.columns]

corr_results = []
for col in num_cols:
    valid = df[[col, 'readmitted_30d']].dropna()
    r, p = pointbiserialr(valid[col].astype(float), valid['readmitted_30d'].astype(float))
    corr_results.append({'feature': col, 'correlation': round(r, 4), 'p_value': round(p, 6)})

corr_df = pd.DataFrame(corr_results).sort_values('correlation', key=abs, ascending=False)
print("Point-biserial correlation with readmitted_30d:")
print(corr_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#4c72b0' if v >= 0 else '#dd8452' for v in corr_df['correlation']]
ax.barh(corr_df['feature'], corr_df['correlation'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Point-Biserial Correlation')
ax.set_title('Numeric Feature Correlation with Readmission')
plt.tight_layout()
plt.savefig('outputs/eda_plots/numeric_correlations.png', dpi=150)
plt.close()
print("\nSaved: outputs/eda_plots/numeric_correlations.png")

Class balance ratio (majority / minority): 4.06:1
  Class 0: 19,177
  Class 1: 4,719

Point-biserial correlation with readmitted_30d:
               feature  correlation  p_value
prior_admissions_count       0.1725 0.000000
              los_days       0.0812 0.000000
      age_at_admission      -0.0015 0.813691

Saved: outputs/eda_plots/numeric_correlations.png


## Section 9: EDA Summary Report

In [9]:
non_ccsr_set = {
    'subject_id', 'hadm_id', 'gender', 'age_at_admission',
    'admission_type', 'insurance', 'race', 'discharge_location',
    'marital_status', 'los_days', 'came_via_ed',
    'prior_admissions_count', 'days_to_next', 'readmitted_30d'
}
_ccsr_cols = [c for c in df.columns if c not in non_ccsr_set]
_clinical_cols = [c for c in df.columns if c not in non_ccsr_set and c not in ('subject_id', 'hadm_id', 'readmitted_30d')]

_missing = df.isnull().sum()
_missing = _missing[_missing > 0]

_vc = df['readmitted_30d'].value_counts().sort_index()
_n0 = int(_vc.get(0, 0))
_n1 = int(_vc.get(1, 0))
_rate = df['readmitted_30d'].mean() * 100

_empty_ccsr = [c for c in _ccsr_cols if df[c].sum() == 0]
_prev = df[_ccsr_cols].mean().sort_values(ascending=False)
_top3 = _prev.head(3)

_ccsr_quoted = [c for c in _ccsr_cols if c.startswith("'") or c.endswith("'")]
_died_rows = int((df['discharge_location'] == 'DIED').sum()) if 'discharge_location' in df.columns else 0
_race_unique = df['race'].nunique() if 'race' in df.columns else 'N/A'

print("=" * 60)
print("DATASET SUMMARY")
print(f"  Total rows              : {len(df):,}")
print(f"  Total columns           : {df.shape[1]}")
print(f"  Clinical feature columns: {len(_clinical_cols)}")
print(f"  CCSR diagnosis columns  : {len(_ccsr_cols)}")
print(f"  Target                  : readmitted_30d")
print()
print("MISSING VALUES")
if len(_missing) == 0:
    print("  None found.")
else:
    for col, cnt in _missing.items():
        print(f"  {col}: {cnt:,} missing ({cnt/len(df)*100:.2f}%)")
print()
print("TARGET")
print(f"  Not readmitted (0): {_n0:,} rows, {_n0/len(df)*100:.1f}%")
print(f"  Readmitted     (1): {_n1:,} rows, {_n1/len(df)*100:.1f}%")
print(f"  Readmission rate  : {_rate:.2f}%")
print()
print("CCSR FEATURES")
print(f"  Total CCSR columns         : {len(_ccsr_cols)}")
print(f"  Empty CCSR columns (all 0) : {len(_empty_ccsr)}")
print(f"  Top 3 most prevalent conditions:")
for cat, prev in _top3.items():
    print(f"    {cat}  ({prev*100:.2f}%)")
print()
print("ISSUES TO FIX IN PREPROCESSING")

issues = []
for col, cnt in _missing.items():
    issues.append(f"  - Missing values in '{col}': {cnt:,} rows -> impute or drop")

if len(_ccsr_quoted) > 0:
    issues.append(f"  - CCSR column names have embedded single quotes ({len(_ccsr_quoted)} cols) -> strip quotes and spaces in preprocessing")

if _died_rows > 0:
    issues.append(f"  - discharge_location = 'DIED': {_died_rows:,} rows -> consider excluding (patient deceased, readmission not applicable)")

issues.append(f"  - race has {_race_unique} unique values -> consolidate rare categories before encoding")

for issue in issues:
    print(issue)

print("=" * 60)

DATASET SUMMARY
  Total rows              : 23,896
  Total columns           : 719
  Clinical feature columns: 705
  CCSR diagnosis columns  : 705
  Target                  : readmitted_30d

MISSING VALUES
  insurance: 403 missing (1.69%)
  discharge_location: 6,577 missing (27.52%)
  marital_status: 510 missing (2.13%)
  days_to_next: 9,783 missing (40.94%)

TARGET
  Not readmitted (0): 19,177 rows, 80.3%
  Readmitted     (1): 4,719 rows, 19.7%
  Readmission rate  : 19.75%

CCSR FEATURES
  Total CCSR columns         : 705
  Empty CCSR columns (all 0) : 0
  Top 3 most prevalent conditions:
    'XXX000'  (58.59%)
    '259  '  (39.73%)
    '98   '  (37.20%)

ISSUES TO FIX IN PREPROCESSING
  - Missing values in 'insurance': 403 rows -> impute or drop
  - Missing values in 'discharge_location': 6,577 rows -> impute or drop
  - Missing values in 'marital_status': 510 rows -> impute or drop
  - Missing values in 'days_to_next': 9,783 rows -> impute or drop
  - CCSR column names have embedded

## Section 10: Three-Class Outcome Analysis

In [ ]:
import pandas as pd
import os

# Load df if Section 1 has not been run
if 'df' not in dir() or not isinstance(df, pd.DataFrame):
    df = pd.read_parquet('outputs/final_dataset.parquet')
    os.makedirs('outputs/eda_plots', exist_ok=True)
    print("df loaded from outputs/final_dataset.parquet")

# Step 1: Confirm or build the three outcome classes
# Check if hospital_expire_flag is already in df
if 'hospital_expire_flag' not in df.columns:
    import duckdb
    con = duckdb.connect()
    flags = con.execute("""
        SELECT hadm_id, hospital_expire_flag
        FROM 'data/admissions.csv'
    """).df()
    df = df.merge(flags, on='hadm_id', how='left')
    print("hospital_expire_flag joined from admissions.csv")
else:
    print("hospital_expire_flag already in df")

# Build outcome class column
# Class 0: not readmitted and did not die
# Class 1: readmitted within 30 days
# Class 2: died in hospital
df['outcome_class'] = df.apply(
    lambda row: 2 if row['hospital_expire_flag'] == 1
    else (1 if row['readmitted_30d'] == 1 else 0),
    axis=1
)

labels = {0: 'Not Readmitted', 1: 'Readmitted', 2: 'Died'}
counts = df['outcome_class'].value_counts().sort_index()
for k, v in counts.items():
    pct = v / len(df) * 100
    print(f"Class {k} ({labels[k]}): {v:,} rows ({pct:.1f}%)")

In [ ]:
# Step 2: Identify CCSR columns
non_ccsr = {
    'subject_id', 'hadm_id', 'gender', 'age_at_admission',
    'admission_type', 'insurance', 'race', 'discharge_location',
    'marital_status', 'los_days', 'came_via_ed',
    'prior_admissions_count', 'days_to_next', 'readmitted_30d',
    'hospital_expire_flag', 'outcome_class'
}
ccsr_cols = [c for c in df.columns if c not in non_ccsr]
print(f"CCSR columns available: {len(ccsr_cols)}")

# Step 3: For each outcome class, get top 5 CCSR conditions by count
# A condition counts if the patient has a 1 in that column
results = {}
for cls in [0, 1, 2]:
    subset = df[df['outcome_class'] == cls][ccsr_cols]
    top5 = subset.sum().sort_values(ascending=False).head(5)
    results[cls] = top5
    print(f"\nTop 5 conditions for Class {cls} ({labels[cls]}):")
    for cat, cnt in top5.items():
        pct = cnt / len(subset) * 100
        print(f"  {cat}: {cnt:,} ({pct:.1f}%)")

In [ ]:
# Step 4: Build comparison bar chart for top conditions
# Combine all unique top-5 categories across all three classes
import matplotlib.pyplot as plt
import numpy as np

all_top_cats = []
for cls in [0, 1, 2]:
    all_top_cats.extend(results[cls].index.tolist())
top_cats = list(dict.fromkeys(all_top_cats))

# Build rate table: for each category, prevalence per class
rate_data = {}
for cls in [0, 1, 2]:
    subset = df[df['outcome_class'] == cls]
    rates = subset[top_cats].mean() * 100
    rate_data[labels[cls]] = rates

import pandas as pd
rate_df = pd.DataFrame(rate_data, index=top_cats)
print("\nPrevalence (%) by outcome class:")
print(rate_df.round(1).to_string())

# Plot grouped bar chart
x = np.arange(len(top_cats))
width = 0.25
colors = ['steelblue', 'tomato', 'gray']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (cls_label, color) in enumerate(zip(labels.values(), colors)):
    ax.bar(x + i * width, rate_df[cls_label], width,
           label=cls_label, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(top_cats, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Top CCSR Conditions by Outcome Class')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_plots/three_class_top_conditions.png', dpi=150)
plt.close()
print("\nSaved: outputs/eda_plots/three_class_top_conditions.png")

In [ ]:
# Step 5: Print summary for professor review
print("=" * 60)
print("THREE-CLASS OUTCOME SUMMARY")
print("=" * 60)
for cls in [0, 1, 2]:
    subset = df[df['outcome_class'] == cls]
    print(f"\n{labels[cls]} (n={len(subset):,}):")
    for cat, cnt in results[cls].items():
        pct = cnt / len(subset) * 100
        print(f"  {cat}: {cnt:,} ({pct:.1f}%)")
print("=" * 60)
print("Chart saved to outputs/eda_plots/three_class_top_conditions.png")